In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# ==========================================
# 1. Environment & A100 Hardware Setup
# ==========================================
# Target NVIDIA A100 GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Targeting device: {device} \n")

# Hyperparameters for Geopolitical Economy
N_CONSUMERS = 3    # 0: OPEC, 1: EU/Asia, 2: USA/Global SPR
N_COMMODITIES = 2  # 0: Oil, 1: General Goods/Capital
N_ASSETS = 1       # 1: Global Financial Asset
N_STATES = 3       # 0: Normal, 1: Blockade Shock, 2: SPR Release

class ComplexGeopoliticalEconomy:
    def __init__(self):
        self.theta = torch.tensor([
            [0.2, 0.8], # OPEC
            [0.8, 0.2], # EU/Asia
            [0.3, 0.7]  # USA/SPR
        ], device=device)
        self.R = torch.tensor([1.0, 0.2, 0.8], device=device)

    def get_endowment(self, state_idx):
        if state_idx == 0:
            return torch.tensor([[1.0, 0.2], [0.1, 1.0], [0.6, 0.6]], device=device)
        elif state_idx == 1:
            return torch.tensor([[0.4, 0.2], [0.1, 1.0], [0.6, 0.6]], device=device)
        else:
            return torch.tensor([[0.4, 0.2], [0.1, 1.0], [1.2, 0.6]], device=device)

    def utility(self, consumption, consumer_idx):
        c = torch.clamp(consumption, min=1e-5)
        return (c[:, 0] ** self.theta[consumer_idx, 0]) * (c[:, 1] ** self.theta[consumer_idx, 1])

env = ComplexGeopoliticalEconomy()

# ==========================================
# 2. Build the SINDy-Inspired Sparse Policy MLP
# ==========================================
class SparsePolicyMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.W1 = nn.Parameter(torch.randn(in_dim, hidden_dim) * 0.1)
        self.b1 = nn.Parameter(torch.zeros(hidden_dim))
        self.W2 = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.1)
        self.b2 = nn.Parameter(torch.zeros(hidden_dim))
        self.W_out = nn.Parameter(torch.randn(hidden_dim, out_dim) * 0.1)
        self.b_out = nn.Parameter(torch.zeros(out_dim))

    def forward(self, x):
        h1 = F.relu(torch.matmul(x, self.W1) + self.b1)
        h2 = F.relu(torch.matmul(h1, self.W2) + self.b2)
        return torch.matmul(h2, self.W_out) + self.b_out

    def get_l1_norm(self):
        return torch.sum(torch.abs(self.W1)) + torch.sum(torch.abs(self.W2)) + torch.sum(torch.abs(self.W_out))

    def apply_hard_threshold(self, threshold=1e-2):
        with torch.no_grad():
            self.W1.data *= (torch.abs(self.W1) > threshold).float()
            self.W2.data *= (torch.abs(self.W2) > threshold).float()
            self.W_out.data *= (torch.abs(self.W_out) > threshold).float()

class GeneratorPolicy(nn.Module):
    def __init__(self):
        super().__init__()
        out_size = N_COMMODITIES + N_ASSETS + (N_CONSUMERS * N_COMMODITIES) + (N_CONSUMERS * N_ASSETS)
        self.mlp = SparsePolicyMLP(N_STATES, 64, out_size)

    def forward(self, state_one_hot):
        logits = self.mlp(state_one_hot)
        p = F.softmax(logits[:, :N_COMMODITIES], dim=-1)
        q = F.softplus(logits[:, N_COMMODITIES : N_COMMODITIES+N_ASSETS])
        idx = N_COMMODITIES + N_ASSETS
        x = F.softplus(logits[:, idx : idx + N_CONSUMERS * N_COMMODITIES]).view(-1, N_CONSUMERS, N_COMMODITIES)
        idx += N_CONSUMERS * N_COMMODITIES
        y = torch.tanh(logits[:, idx : idx + N_CONSUMERS * N_ASSETS]).view(-1, N_CONSUMERS, N_ASSETS)
        return p, q, x, y

class Adversary(nn.Module):
    def __init__(self):
        super().__init__()
        in_dim = N_STATES + N_COMMODITIES + N_ASSETS + (N_CONSUMERS * N_COMMODITIES) + (N_CONSUMERS * N_ASSETS)
        self.net = nn.Sequential(nn.Linear(in_dim, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU())
        self.out = nn.Linear(64, N_CONSUMERS * N_COMMODITIES + N_CONSUMERS * N_ASSETS)

    def forward(self, state_one_hot, p, q, x_gen, y_gen):
        gen_features = torch.cat([p, q, x_gen.view(-1, N_CONSUMERS*N_COMMODITIES), y_gen.view(-1, N_CONSUMERS*N_ASSETS)], dim=-1)
        logits = self.out(self.net(torch.cat([state_one_hot, gen_features], dim=-1)))
        x_dev = F.softplus(logits[:, :N_CONSUMERS * N_COMMODITIES]).view(-1, N_CONSUMERS, N_COMMODITIES)
        y_dev = torch.tanh(logits[:, N_CONSUMERS * N_COMMODITIES:]).view(-1, N_CONSUMERS, N_ASSETS)
        return x_dev, y_dev

gen = GeneratorPolicy().to(device)
adv = Adversary().to(device)
opt_gen = optim.Adam(gen.parameters(), lr=1e-3)
opt_adv = optim.Adam(adv.parameters(), lr=5e-3)

LAMBDA_L1 = 0.005
states = torch.eye(N_STATES, device=device)

print("Training SINDy-Regularized GAPNet...")
for epoch in range(3000):

    # =======================================================
    # PHASE 1: Train Adversary (Maximize Exploitability)
    # =======================================================
    opt_adv.zero_grad()

    # 1. Forward pass Generator
    p, q, x_gen, y_gen = gen(states)

    # 2. Forward pass Adversary (DETACH Generator outputs!)
    x_dev, y_dev = adv(states, p.detach(), q.detach(), x_gen.detach(), y_gen.detach())

    adv_budget_penalty = 0
    exploitability = 0

    for s_idx in range(N_STATES):
        endowments = env.get_endowment(s_idx)
        for c_idx in range(N_CONSUMERS):
            # Use detached prices for adversary calculations
            wealth = torch.sum(endowments[c_idx] * p[s_idx].detach())
            dev_cost = torch.sum(x_dev[s_idx, c_idx] * p[s_idx].detach()) + torch.sum(y_dev[s_idx, c_idx] * q[s_idx].detach())

            adv_budget_penalty += F.relu(dev_cost - wealth)**2

            gen_u = env.utility(x_gen[s_idx, c_idx].detach().unsqueeze(0), c_idx).squeeze()
            dev_u = env.utility(x_dev[s_idx, c_idx].unsqueeze(0), c_idx).squeeze()

            exploitability += (dev_u - gen_u)

    # Maximize exploitability = Minimize negative exploitability
    loss_adv = -exploitability + 100 * adv_budget_penalty
    loss_adv.backward()
    opt_adv.step()


    # =======================================================
    # PHASE 2: Train Generator (Minimize Exploitability)
    # =======================================================
    opt_gen.zero_grad()

    # 1. Forward pass Generator again (new graph)
    p, q, x_gen, y_gen = gen(states)

    # 2. Forward pass Adversary (DO NOT DETACH - we need gradients to flow back to Gen)
    x_dev, y_dev = adv(states, p, q, x_gen, y_gen)

    gen_budget_penalty = 0
    walras_penalty = 0
    exploitability_gen = 0

    for s_idx in range(N_STATES):
        endowments = env.get_endowment(s_idx)
        walras_penalty += torch.sum((torch.sum(x_gen[s_idx], dim=0) - torch.sum(endowments, dim=0))**2) + torch.sum(y_gen[s_idx]**2)

        for c_idx in range(N_CONSUMERS):
            wealth = torch.sum(endowments[c_idx] * p[s_idx])
            gen_cost = torch.sum(x_gen[s_idx, c_idx] * p[s_idx]) + torch.sum(y_gen[s_idx, c_idx] * q[s_idx])

            gen_budget_penalty += F.relu(gen_cost - wealth)**2

            gen_u = env.utility(x_gen[s_idx, c_idx].unsqueeze(0), c_idx).squeeze()
            dev_u = env.utility(x_dev[s_idx, c_idx].unsqueeze(0), c_idx).squeeze()

            exploitability_gen += (dev_u - gen_u)

    loss_gen = exploitability_gen + 100 * walras_penalty + 100 * gen_budget_penalty + LAMBDA_L1 * gen.mlp.get_l1_norm()
    loss_gen.backward()
    opt_gen.step()

    # =======================================================
    # SINDy Thresholding
    # =======================================================
    # Applied after the optimizer step
    if epoch > 1000 and epoch % 500 == 0:
        gen.mlp.apply_hard_threshold(threshold=0.01)

    # Optional: Print progress
    if epoch % 500 == 0:
        print(f"Epoch {epoch} | Loss Gen: {loss_gen.item():.4f} | Loss Adv: {loss_adv.item():.4f}")

with torch.no_grad():
    p, q, x, y = gen(states)
print("Training Complete.")

Targeting device: cuda 

Training SINDy-Regularized GAPNet...
Epoch 0 | Loss Gen: 184.7039 | Loss Adv: 35.2424
Epoch 500 | Loss Gen: 2.4095 | Loss Adv: -1.1225
Epoch 1000 | Loss Gen: 1.4783 | Loss Adv: -0.5247
Epoch 1500 | Loss Gen: 1.0687 | Loss Adv: -0.3331
Epoch 2000 | Loss Gen: 0.8661 | Loss Adv: -0.2767
Epoch 2500 | Loss Gen: 0.7414 | Loss Adv: -0.2604
Training Complete.


In [7]:
class TrueSINDyPolicy(nn.Module):
    def __init__(self, continuous_state_dim, out_dim, poly_degree=2):
        super().__init__()
        self.poly_degree = poly_degree

        # Calculate size of the library: 1 + states + states^2 + cross-terms...
        # For a 3D state and degree 2, library size is 10.
        self.library_size = self._calc_library_size(continuous_state_dim, poly_degree)

        # Xi is the sparse coefficient matrix. No hidden layers!
        # It maps the algebraic basis directly to the economic outputs (prices, flows)
        self.Xi = nn.Parameter(torch.randn(self.library_size, out_dim) * 0.1)

    def _calc_library_size(self, dim, degree):
        from math import comb
        return sum(comb(dim + d - 1, d) for d in range(degree + 1))

    def build_basis_library(self, state):
        """
        Constructs Theta(X). If state is [s1, s2]:
        Returns: [1, s1, s2, s1^2, s2^2, s1*s2]
        """
        batch_size = state.shape[0]
        # 1. Constant term
        library = [torch.ones(batch_size, 1, device=state.device)]

        # 2. Linear terms
        library.append(state)

        # 3. Quadratic terms (and cross terms)
        if self.poly_degree >= 2:
            for i in range(state.shape[1]):
                for j in range(i, state.shape[1]):
                    library.append((state[:, i] * state[:, j]).unsqueeze(1))

        return torch.cat(library, dim=1)

    def forward(self, state):
        # 1. Expand continuous state into polynomial basis Theta(S)
        Theta_S = self.build_basis_library(state)

        # 2. Multiply by sparse coefficients Xi
        # Output = Theta(S) * Xi
        logits = torch.matmul(Theta_S, self.Xi)
        return logits

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# ==========================================
# 1. Environment Setup (Continuous States)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_CONSUMERS = 3
N_COMMODITIES = 2
N_ASSETS = 1
STATE_DIM = 2      # e.g., [supply_shock, liquidity_index] continuous from 0 to 1

class ContinuousGeopoliticalEconomy:
    def __init__(self):
        self.theta = torch.tensor([
            [0.2, 0.8], # OPEC
            [0.8, 0.2], # EU/Asia
            [0.3, 0.7]  # USA/SPR
        ], device=device)

    def get_endowment(self, states):
        """
        States is shape [batch_size, 2].
        Endowments shift continuously based on the state variables.
        """
        batch_size = states.shape[0]
        supply_shock = states[:, 0].unsqueeze(1) # 0 = Normal, 1 = Severe shock

        # Base endowments [Consumer, Commodity]
        base = torch.tensor([[1.0, 0.2], [0.1, 1.0], [0.6, 0.6]], device=device)
        base = base.unsqueeze(0).repeat(batch_size, 1, 1)

        # Shock reduces Consumer 0's first commodity (e.g., Oil)
        shock_effect = torch.zeros_like(base)
        shock_effect[:, 0, 0] = -0.6 * supply_shock.squeeze()

        return base + shock_effect

    def utility(self, consumption, consumer_idx):
        c = torch.clamp(consumption, min=1e-5)
        return (c[:, 0] ** self.theta[consumer_idx, 0]) * (c[:, 1] ** self.theta[consumer_idx, 1])

env = ContinuousGeopoliticalEconomy()

# ==========================================
# 2. True SINDy Policy (Generator)
# ==========================================
class TrueSINDyPolicy(nn.Module):
    def __init__(self, in_dim, out_dim, poly_degree=2):
        super().__init__()
        self.poly_degree = poly_degree
        self.library_size = self._calc_library_size(in_dim, poly_degree)

        # Sparse coefficient matrix (No hidden layers!)
        self.Xi = nn.Parameter(torch.randn(self.library_size, out_dim) * 0.1)

    def _calc_library_size(self, dim, degree):
        from math import comb
        return sum(comb(dim + d - 1, d) for d in range(degree + 1))

    def build_basis_library(self, state):
        batch_size = state.shape[0]
        library = [torch.ones(batch_size, 1, device=state.device)]
        library.append(state)

        if self.poly_degree >= 2:
            for i in range(state.shape[1]):
                for j in range(i, state.shape[1]):
                    library.append((state[:, i] * state[:, j]).unsqueeze(1))
        return torch.cat(library, dim=1)

    def forward(self, state):
        Theta_S = self.build_basis_library(state)
        return torch.matmul(Theta_S, self.Xi)

class GeneratorPolicy(nn.Module):
    def __init__(self):
        super().__init__()
        out_size = N_COMMODITIES + N_ASSETS + (N_CONSUMERS * N_COMMODITIES) + (N_CONSUMERS * N_ASSETS)
        self.sindy = TrueSINDyPolicy(STATE_DIM, out_size, poly_degree=2)

    def forward(self, states):
        logits = self.sindy(states)
        p = F.softmax(logits[:, :N_COMMODITIES], dim=-1)
        q = F.softplus(logits[:, N_COMMODITIES : N_COMMODITIES+N_ASSETS])
        idx = N_COMMODITIES + N_ASSETS

        x = F.softplus(logits[:, idx : idx + N_CONSUMERS * N_COMMODITIES]).view(-1, N_CONSUMERS, N_COMMODITIES)
        idx += N_CONSUMERS * N_COMMODITIES
        y = torch.tanh(logits[:, idx : idx + N_CONSUMERS * N_ASSETS]).view(-1, N_CONSUMERS, N_ASSETS)
        return p, q, x, y

# ==========================================
# 3. Adversary (Standard MLP)
# ==========================================
class Adversary(nn.Module):
    def __init__(self):
        super().__init__()
        in_dim = STATE_DIM + N_COMMODITIES + N_ASSETS + (N_CONSUMERS * N_COMMODITIES) + (N_CONSUMERS * N_ASSETS)
        self.net = nn.Sequential(nn.Linear(in_dim, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU())
        self.out = nn.Linear(64, N_CONSUMERS * N_COMMODITIES + N_CONSUMERS * N_ASSETS)

    def forward(self, states, p, q, x_gen, y_gen):
        gen_features = torch.cat([p, q, x_gen.view(-1, N_CONSUMERS*N_COMMODITIES), y_gen.view(-1, N_CONSUMERS*N_ASSETS)], dim=-1)
        logits = self.out(self.net(torch.cat([states, gen_features], dim=-1)))

        x_dev = F.softplus(logits[:, :N_CONSUMERS * N_COMMODITIES]).view(-1, N_CONSUMERS, N_COMMODITIES)
        y_dev = torch.tanh(logits[:, N_CONSUMERS * N_COMMODITIES:]).view(-1, N_CONSUMERS, N_ASSETS)
        return x_dev, y_dev

# ==========================================
# 4. INITIALIZATION
# ==========================================
gen = GeneratorPolicy().to(device)
adv = Adversary().to(device)

opt_gen = optim.Adam(gen.parameters(), lr=1e-3)
opt_adv = optim.Adam(adv.parameters(), lr=5e-3)

LAMBDA_L2 = 0.001   # Ridge penalty for SINDy
STRR_THRESHOLD = 0.05 # Threshold for masking
BATCH_SIZE = 64

print("Training True SINDy GAPNet...")
for epoch in range(3000):
    # Sample continuous states for this epoch [Batch, 2]
    states = torch.rand(BATCH_SIZE, STATE_DIM, device=device)
    endowments = env.get_endowment(states) # Shape: [Batch, Consumers, Commodities]

    # =======================================================
    # PHASE 1: Train Adversary (Maximize Exploitability)
    # =======================================================
    opt_adv.zero_grad()
    p, q, x_gen, y_gen = gen(states)

    # Detach Generator outputs to prevent gradient contamination
    x_dev, y_dev = adv(states, p.detach(), q.detach(), x_gen.detach(), y_gen.detach())

    adv_budget_penalty = 0
    exploitability = 0

    for c_idx in range(N_CONSUMERS):
        wealth = torch.sum(endowments[:, c_idx, :] * p.detach(), dim=1)
        dev_cost = torch.sum(x_dev[:, c_idx, :] * p.detach(), dim=1) + torch.sum(y_dev[:, c_idx, :] * q.detach(), dim=1)

        adv_budget_penalty += torch.mean(F.relu(dev_cost - wealth)**2)

        gen_u = env.utility(x_gen[:, c_idx, :].detach(), c_idx)
        dev_u = env.utility(x_dev[:, c_idx, :], c_idx)

        exploitability += torch.mean(dev_u - gen_u)

    loss_adv = -exploitability + 100 * adv_budget_penalty
    loss_adv.backward()
    opt_adv.step()

    # =======================================================
    # PHASE 2: Train Generator (Minimize Exploitability + STRRidge)
    # =======================================================
    opt_gen.zero_grad()

    # 1. STRRidge Masking (Zero out small coefficients)
    with torch.no_grad():
        small_indices = torch.abs(gen.sindy.Xi) < STRR_THRESHOLD
        gen.sindy.Xi[small_indices] = 0.0

    p, q, x_gen, y_gen = gen(states)
    x_dev, y_dev = adv(states, p, q, x_gen, y_gen)

    gen_budget_penalty = 0
    walras_penalty = 0
    exploitability_gen = 0

    # Walras Law Penalty (aggregate demand <= aggregate supply)
    walras_penalty += torch.mean((torch.sum(x_gen, dim=1) - torch.sum(endowments, dim=1))**2)
    walras_penalty += torch.mean(torch.sum(y_gen, dim=1)**2)

    for c_idx in range(N_CONSUMERS):
        wealth = torch.sum(endowments[:, c_idx, :] * p, dim=1)
        gen_cost = torch.sum(x_gen[:, c_idx, :] * p, dim=1) + torch.sum(y_gen[:, c_idx, :] * q, dim=1)

        gen_budget_penalty += torch.mean(F.relu(gen_cost - wealth)**2)

        gen_u = env.utility(x_gen[:, c_idx, :], c_idx)
        dev_u = env.utility(x_dev[:, c_idx, :], c_idx)

        exploitability_gen += torch.mean(dev_u - gen_u)

    # L2 Ridge Penalty applied only to active parameters
    loss_gen = exploitability_gen + 100 * walras_penalty + 100 * gen_budget_penalty + LAMBDA_L2 * torch.sum(gen.sindy.Xi ** 2)
    loss_gen.backward()

    # 2. Kill gradients for zeroed-out terms so Adam doesn't bring them back
    gen.sindy.Xi.grad[small_indices] = 0.0

    opt_gen.step()

    if epoch > 0 and epoch % 500 == 0:
        active_terms = (~small_indices).sum().item()
        total_terms = gen.sindy.Xi.numel()
        print(f"Epoch {epoch} | Gen Loss: {loss_gen.item():.4f} | Active SINDy Terms: {active_terms}/{total_terms}")

print("Training Complete. The Xi matrix now contains interpretable coefficients!")

Training True SINDy GAPNet...
Epoch 500 | Gen Loss: 9.3240 | Active SINDy Terms: 17/72
Epoch 1000 | Gen Loss: 5.8901 | Active SINDy Terms: 16/72
Epoch 1500 | Gen Loss: 5.5621 | Active SINDy Terms: 14/72
Epoch 2000 | Gen Loss: 5.4415 | Active SINDy Terms: 13/72
Epoch 2500 | Gen Loss: 5.2379 | Active SINDy Terms: 13/72
Training Complete. The Xi matrix now contains interpretable coefficients!


In [10]:
def extract_and_print_equations(generator):
    # 1. Define human-readable names for the polynomial basis features
    feature_names = [
        "1",
        "Shock",
        "Liq",
        "Shock^2",
        "Shock*Liq",
        "Liq^2"
    ]

    # 2. Define human-readable names for the outputs (matching the forward pass slice order)
    output_names = [
        "Price_Oil (logit)",
        "Price_Goods (logit)",
        "Asset_Price (logit)",
        "Flow_OPEC_Oil (logit)",
        "Flow_OPEC_Goods (logit)",
        "Flow_EU_Oil (logit)",
        "Flow_EU_Goods (logit)",
        "Flow_USA_Oil (logit)",
        "Flow_USA_Goods (logit)",
        "Portf_OPEC_Asset (logit)",
        "Portf_EU_Asset (logit)",
        "Portf_USA_Asset (logit)"
    ]

    # Fetch the sparse matrix
    Xi_matrix = generator.sindy.Xi.detach().cpu().numpy()

    print("\n" + "="*70)
    print("   INTERPRETABLE MACROECONOMIC POLICY EQUATIONS (SINDy)")
    print("="*70)

    for out_idx in range(len(output_names)):
        terms = []
        for feat_idx in range(len(feature_names)):
            weight = Xi_matrix[feat_idx, out_idx]

            # STRRidge forced pruned weights to exactly 0.0
            if abs(weight) > 0.0:
                terms.append(f"{weight:+.4f}*{feature_names[feat_idx]}")

        if len(terms) == 0:
            equation = "0.0  (Completely pruned)"
        else:
            # Join terms, replacing a leading "+" for cleaner reading
            equation = " ".join(terms).lstrip('+')

        print(f"{output_names[out_idx]:<26} = {equation}")

    print("\n" + "-"*70)
    print("Note: Final actions are derived by applying the network's activations:")
    print(" -> Final Prices       = Softmax( [Price_Oil(logit), Price_Goods(logit)] )")
    print(" -> Final Asset Price  = Softplus( Asset_Price(logit) )")
    print(" -> Final Trade Flows  = Softplus( Flow_*(logit) )")
    print(" -> Final Portfolios   = Tanh( Portf_*(logit) )")
    print("="*70 + "\n")

# Call the function
extract_and_print_equations(gen)


   INTERPRETABLE MACROECONOMIC POLICY EQUATIONS (SINDy)
Price_Oil (logit)          = 0.2479*Liq^2
Price_Goods (logit)        = 0.0  (Completely pruned)
Asset_Price (logit)        = -0.7031*Shock*Liq
Flow_OPEC_Oil (logit)      = -0.7683*Shock*Liq
Flow_OPEC_Goods (logit)    = -0.0550*Shock^2
Flow_EU_Oil (logit)        = -0.5417*Liq
Flow_EU_Goods (logit)      = -0.2215*Liq
Flow_USA_Oil (logit)       = -1.0994*1 -0.7426*Shock*Liq
Flow_USA_Goods (logit)     = -0.5254*1
Portf_OPEC_Asset (logit)   = -0.5256*Shock
Portf_EU_Asset (logit)     = -0.1645*Shock^2
Portf_USA_Asset (logit)    = 0.0980*1 +0.6761*Shock^2

----------------------------------------------------------------------
Note: Final actions are derived by applying the network's activations:
 -> Final Prices       = Softmax( [Price_Oil(logit), Price_Goods(logit)] )
 -> Final Asset Price  = Softplus( Asset_Price(logit) )
 -> Final Trade Flows  = Softplus( Flow_*(logit) )
 -> Final Portfolios   = Tanh( Portf_*(logit) )

